[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/1-2_SMART%EA%B2%BD%EB%B3%B4_%EA%B7%9C%EC%B9%99%EA%B3%BC%EC%84%B1%EB%8A%A5.ipynb)

# 1-2 · SMART 신호로 고장을 미리 알 수 있을까?

**⏱ 80분** · 모듈 1: 현장 설비군의 고장 데이터 → **경보** → 비용

## 🎯 이번 시간의 질문

> 디스크가 스스로 보고하는 상태 값(SMART)으로 **곧 고장 날 디스크를 미리 골라낼 수 있을까요?**

지난 시간에 고장은 드물고(1.4%/년), 사용 연수만으로는 고장 날 디스크를 고르기 어렵다는 것을 보았습니다.
CBM의 핵심 전제는 "**고장 전에 상태 신호가 먼저 변한다**"는 것입니다. 이번 시간에는 가장 단순한 이상탐지인 **규칙 경보**(`값 > 기준`이면 경보)를 만들고,
그 경보가 얼마나 잘 맞는지 **탐지율·경보 적중률·준비 여유시간**으로 평가합니다.

### 사용할 SMART 지표

| 열 | 이름 | 뜻 (정상 디스크는 보통 0) |
|---|---|---|
| `smart_197_raw` | 보류 섹터 수 | 읽기가 불안정해서 "나중에 다시 확인하려고" 보류해 둔 영역 수 |
| `smart_5_raw` | 재할당 섹터 수 | 불량으로 판정되어 예비 영역으로 옮긴 영역 수 |
| `smart_187_raw` | 복구 불가 오류 | 디스크가 스스로 고치지 못한 읽기 오류 수 |
| `smart_198_raw` | 오프라인 복구 불가 | 자체 점검 중 발견한 복구 불가 영역 수 |

**이번 시간에 익힐 패턴:** P2 조건(`>`, `&`, `|`) · 교차표 `pd.crosstab` · P4 선그래프·히스토그램

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/backblaze_2026Q1.csv.gz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
raw = pd.read_csv('data/backblaze_2026Q1.csv.gz',
                  dtype={'serial_number': 'category', 'model': 'category'},
                  parse_dates=['date'])
print('읽기 완료:', raw.shape)

## 1. 고장 난 디스크 한 대의 기록 보기

디스크 `33M0A033FVKG`는 2월 15일에 고장 났습니다. 고장 전에 보류 섹터 수(`smart_197_raw`)가 어떻게 변했는지 봅니다.

**P2 → P4** — 한 디스크의 행만 골라내고, 날짜에 따른 값을 선그래프로 그립니다. `plt.axvline`은 세로선을 긋습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
one = raw[raw['serial_number'] == '33M0A033FVKG']
plt.plot(one['date'], one['smart_197_raw'], marker='o')
plt.axvline(pd.Timestamp('2026-02-15'), color='red', linestyle='--', label='고장일')
plt.xlabel('날짜')
plt.ylabel('보류 섹터 수 (smart_197)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 1월 초에는 0이었다가 **1월 13일부터 보류 섹터가 생기기 시작**해 고장 직전 80개 넘게 늘었습니다.
이 디스크는 고장 **33일 전**에 이상 신호를 보였습니다. 이렇게 "신호가 보이기 시작한 시점(P)"부터 "기능이 멈춘 시점(F)"까지의 구간을 **P–F 구간**이라고 하며, 이 구간이 길수록 정비를 준비할 시간이 많습니다.

하지만 한 대의 예로는 부족합니다. **6만 대 전체**에서 이 신호가 얼마나 잘 맞는지 확인해야 합니다.

## 2. 경보 규칙 만들기

가장 단순한 규칙: **"보류 섹터가 하나라도 생기면(> 0) 경보"**.
아래 코드는 523만 행 각각에 대해 조건이 참/거짓인지 계산합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
signal = raw['smart_197_raw'] > 0
signal.sum()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 조건이 참인 **행**은 약 10만 개입니다. 하지만 우리가 알고 싶은 것은 행이 아니라 **디스크**입니다. 한 디스크가 여러 날 경보를 내도 교체는 한 번입니다.

### ▶ 디스크 단위로 정리하는 함수

아래 함수 `alarm_table`은 디스크마다 다음을 정리해 줍니다. (코드는 읽기만 하세요. 결과 표의 열 뜻만 알면 됩니다.)

| 결과 열 | 뜻 |
|---|---|
| `failed` | 90일 안에 고장 났으면 1 |
| `fail_date` | 고장 날짜 |
| `alarm_date` | **처음** 경보가 난 날짜 (고장 디스크는 고장 **전날까지**의 기록만 사용) |
| `alarm` | 경보가 한 번이라도 났으면 True |
| `lead_days` | 첫 경보 → 고장까지 며칠 여유가 있었나 |

In [ ]:
# ▶ 그대로 실행하세요
def alarm_table(raw, signal):
    """디스크마다 고장 여부, 고장일, 첫 경보일, 경보~고장 여유 일수를 정리한다."""
    drives = raw.groupby('serial_number', observed=True).agg(failed=('failure', 'max'))
    drives['fail_date'] = raw[raw['failure'] == 1].groupby('serial_number', observed=True)['date'].min()
    usable = signal & (raw['failure'] == 0)          # 고장 당일 기록은 '미리 안 것'이 아니므로 제외
    drives['alarm_date'] = raw[usable].groupby('serial_number', observed=True)['date'].min()
    drives['alarm'] = drives['alarm_date'].notna()
    drives['lead_days'] = (drives['fail_date'] - drives['alarm_date']).dt.days
    return drives

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
result = alarm_table(raw, signal)
result[result['failed'] == 1].head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 고장 디스크만 몇 개 보았습니다. `alarm`이 True이고 `lead_days`가 큰 디스크는 **미리 알아낸** 경우, `alarm`이 False인 디스크는 **놓친** 경우입니다.

## 3. 경보는 얼마나 맞았나 — 교차표

`pd.crosstab(행, 열)`은 두 구분을 교차한 **개수 표**를 만듭니다. 실제 고장 여부 × 경보 여부로 네 칸이 나옵니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
pd.crosstab(result['failed'], result['alarm'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 네 칸의 뜻과 개수를 읽어 봅시다.

| | 경보 없음 (False) | 경보 있음 (True) |
|---|---|---|
| **고장 안 남 (0)** | 58,237 · 정상 판단 ✔ | 1,563 · 경보했지만 90일 안에 고장 안 남 |
| **고장 남 (1)** | 80 · **놓침** ✘ | 120 · **미리 탐지** ✔ |

이 표에서 경보 성능을 나타내는 두 숫자를 계산합니다.

- **탐지율(재현율, Recall)** = 미리 탐지 ÷ 전체 고장 → "고장 중 몇 %를 미리 잡았나"
- **경보 적중률(정밀도, Precision)** = 미리 탐지 ÷ 전체 경보 → "경보한 디스크 중 몇 %가 실제로 고장 났나"

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
hits = ((result['failed'] == 1) & result['alarm']).sum()
n_alarms = result['alarm'].sum()
n_failed = result['failed'].sum()
print('탐지율:', round(hits / n_failed * 100, 1), '%')
print('경보 적중률:', round(hits / n_alarms * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 탐지율 60.0%, 경보 적중률 7.1%.

💬 **결과 해설**
- 고장 200건 중 **120건(60%)**은 고장 전에 신호가 있었습니다. 나머지 40%는 이 신호 없이 갑자기 고장 났습니다.
- 경보를 낸 1,683대 중 90일 안에 실제 고장 난 것은 **7.1%**뿐입니다. 경보 14대 중 1대꼴입니다.
- 7.1%가 낮아 보이지만, 아무 디스크나 고르면 고장 날 확률은 200 ÷ 60,000 = **0.33%**입니다. 경보가 난 디스크는 **약 21배** 더 위험합니다.
- 고장 자체가 드물기 때문에, 좋은 신호라도 경보 적중률은 낮게 나오기 쉽습니다. 이것이 **드문 고장을 경보할 때의 근본적인 어려움**입니다.

## 4. 얼마나 미리 알았나 — 준비 여유시간

경보가 고장 **하루 전**에 나면 교체 준비가 어렵습니다. 미리 탐지한 120대의 `lead_days`(첫 경보 → 고장) 분포를 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
detected = result[(result['failed'] == 1) & result['alarm']]
plt.hist(detected['lead_days'], bins=18)
plt.axvline(7, color='red', linestyle='--', label='7일')
plt.xlabel('첫 경보부터 고장까지 (일)')
plt.ylabel('디스크 수')
plt.legend()
plt.show()
print('7일 이상 여유:', (detected['lead_days'] >= 7).sum(), '대')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 여유 일수의 중앙값은 약 38일이고, **120대 중 106대는 7일 이상** 여유가 있었습니다. 교체 디스크를 주문·배치하는 데 1주일이면 충분한 현장이라면, 이 106건은 고장 전에 계획 교체할 수 있었던 셈입니다.

⚠️ 경보 디스크의 약 2/3는 **관찰 첫날(1월 1일)에 이미** 보류 섹터가 있었습니다. 실제 신호는 그보다 먼저 시작됐을 수 있으므로, 이 데이터의 여유 일수는 **실제보다 짧게** 계산된 값입니다.

### ▶ 요약 함수

같은 계산을 조건만 바꿔 여러 번 할 것이므로, 3~4절의 계산을 함수 하나로 묶어 두었습니다.

In [ ]:
# ▶ 그대로 실행하세요
def summarize(result, lead=7):
    """경보 결과 표를 한 줄 요약: 경보 대수, 탐지, 탐지율, 적중률, lead일 이상 여유."""
    hits = ((result['failed'] == 1) & result['alarm']).sum()
    n_alarms = result['alarm'].sum()
    return pd.Series({'경보 대수': n_alarms,
                      '미리 탐지': hits,
                      '탐지율(%)': round(hits / result['failed'].sum() * 100, 1),
                      '적중률(%)': round(hits / n_alarms * 100, 1),
                      f'{lead}일 이상 여유': (result['lead_days'] >= lead).sum()})

summarize(result)

---
## 5. 실습 과제 (35분)

### 📝 과제 1 · 경보 기준을 바꾸면? (기준값 실험)
기준을 `> 0`에서 `> 8`, `> 16`, `> 64`로 올려 가며 아래 표를 채우세요. **실행 전에** 경보 대수와 탐지율이 늘지 줄지 먼저 예상해 보세요.

| 기준 | 경보 대수 | 미리 탐지 | 탐지율(%) | 적중률(%) | 7일 이상 여유 |
|---|---|---|---|---|---|
| > 0 | 1,683 | 120 | 60.0 | 7.1 | 106 |
| > 8 | | | | | |
| > 16 | | | | | |
| > 64 | | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: summarize(alarm_table(raw, raw['smart_197_raw'] > 8)) 처럼 한 줄로 계산합니다.


✏️ **나의 답:** 기준을 올리면 경보 대수·탐지율·적중률이 각각 어떻게 변했나요? 왜 그럴까요?

### 📝 과제 2 · 어느 SMART 지표가 가장 좋은 신호인가?
`smart_5_raw`, `smart_187_raw`, `smart_198_raw` 각각에 대해 `> 0` 규칙의 성능을 구하고 `smart_197_raw`와 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 과제 1의 코드에서 열 이름만 바꿉니다. for col in ['smart_5_raw', ...]: 로 반복해도 됩니다.


✏️ **나의 답:** 탐지율이 가장 높은 지표와 적중률이 가장 높은 지표는 각각 무엇인가요? 하나만 고른다면?

### 📝 과제 3 · 두 신호를 함께 쓰면? (도전)
"보류 섹터 > 0 **또는** 재할당 섹터 > 0"이면 경보하는 규칙을 만들어 보세요. 파이썬에서 **또는**은 `|`, **그리고**는 `&`이며, 각 조건을 괄호로 감쌉니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: signal2 = (raw['smart_197_raw'] > 0) | (raw['smart_5_raw'] > 0)


✏️ **나의 답:** 탐지율은 얼마나 올랐고, 대신 무엇을 잃었나요?

### 📝 과제 4 · 생각해 보기 — 1,563대는 정말 "헛경보"일까?
경보가 났지만 90일 안에 고장 나지 않은 디스크 1,563대를 **헛경보(오경보)**라고 단정할 수 있을까요? 이 데이터의 관찰 기간(90일)을 생각하며 답하세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 규칙 경보 | `값 > 기준`이면 경보. 가장 단순한 이상탐지 |
| 평가 단위 | 행이 아니라 **디스크(설비) 단위**로, 처음 경보한 날을 기준으로 |
| 탐지율 vs 적중률 | 60% vs 7.1%. 고장이 드물면 적중률은 낮게 나온다. 그래도 무작위보다 21배 효율적 |
| 여유시간 | 경보는 **미리** 나야 쓸모 있다. 7일 이상 여유 106건 |
| 교환 관계 | 기준을 올리면 헛경보↓ 탐지율↓. 최선의 기준은 **비용**이 정한다 |

다음: **1-3 · 경보 기준을 비용으로 정하기**